# Screen-tuned SmolVLM fork (stretch experiment)\n\nLoRA fine-tune of SmolVLM-256M-Instruct on ScreenQA for screen Q&A.\nPart of Pawline experiments/screen-fork - see RESEARCH.md for the honest\nfeasibility notes (the OCR hybrid beats stock SmolVLM on text questions;\nthis fork targets VISUAL questions where OCR is blind). NOT a shipped\ndefault: the result must beat stock SmolVLM AND the OCR hybrid on the\nPawline battery (docs/EXPERIMENTS.md 4d/4e) before it is used anywhere.\n\n**Runtime**: Colab free tier: Runtime > Change runtime type > T4 GPU.\nKaggle: enable GPU (P100) in the notebook settings.\n\n**Licenses**: SmolVLM Apache-2.0; ScreenQA annotations CC BY 4.0\n(attribute Google); the Rico screenshots behind ScreenQA carry Rico's\nown research terms (interactionmining.org/archive/rico/copyright.txt) -\nread them before releasing anything trained on this. If you add WebSight\nto the mix: CC-BY-4.0 + you must disclose its use when releasing a model.\nKeep this notice in any released model card.\n

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes Pillow\n

In [ ]:
import torch, datasets\nprint(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - stop here')\n

## Data: ScreenQA train split\n\n86K human-annotated QA over Rico mobile screenshots (CC BY 4.0). Mobile,\nnot desktop - the known domain gap from RESEARCH.md. We cap the training\nsubset so a free-tier run finishes in the session; raise N if you have\nKaggle's 30 h/week.\n

In [ ]:
N = 6000  # raise toward 20000 on Kaggle; full train is 68,951\nds = datasets.load_dataset('bevaya/RICO-ScreenQA', split='train').shuffle(seed=0).select(range(N))\nds\n

In [ ]:
def to_chat(ex):\n    return {'messages': [\n        {'role':'user','content':[{'type':'image'},{'type':'text','text':ex['question']}]},\n        {'role':'assistant','content':[{'type':'text','text':ex['ground_truth'][0]}]},\n    ], 'image': ex['image']}\nds = ds.map(to_chat, remove_columns=[c for c in ds.column_names if c not in ('messages','image')])\n

## Model + LoRA\n\nVision tower frozen, LoRA on the language backbone (r=32). 4-bit base\nkeeps it inside the free T4.\n

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig\nfrom peft import LoraConfig, get_peft_model\nBASE = 'HuggingFaceTB/SmolVLM-256M-Instruct'\nproc = AutoProcessor.from_pretrained(BASE)\nmodel = AutoModelForVision2Seq.from_pretrained(BASE, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map='auto')\nmodel = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], task_type=None))\nmodel.print_trainable_parameters()\n

## Train\n

In [ ]:
from trl import SFTConfig, SFTTrainer\ndef collate(batch):\n    texts = [proc.apply_chat_template(b['messages'], add_generation_prompt=False) for b in batch]\n    images = [[b['image']] for b in batch]\n    out = proc(text=texts, images=images, return_tensors='pt', padding=True)\n    out['labels'] = out['input_ids'].clone()\n    return out\ncfg = SFTConfig(output_dir='./smolvlm-screen-lora', per_device_train_batch_size=2, gradient_accumulation_steps=16,\n    num_train_epochs=1, learning_rate=1e-4, logging_steps=20, save_steps=500, bf16=True,\n    gradient_checkpointing=True, max_length=2048, dataset_num_proc=1, report_to=[])\ntrainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, data_collator=collate)\ntrainer.train()\ntrainer.save_model()\n

## Merge + export\n\nAfter this notebook: convert to GGUF with llama.cpp's converter (CPU,\nruns anywhere), then re-run the Pawline battery (docs/EXPERIMENTS.md\n4d/4e) against stock 256M/500M and the OCR hybrid. Ship only if it wins.\n

In [ ]:
merged = model.merge_and_unload()  # needs enough RAM; on Colab this is the tight step - restart and merge on CPU if it OOMs\nmerged.save_pretrained('./smolvlm-screen-merged')\nproc.save_pretrained('./smolvlm-screen-merged')\nprint('then: python convert_hf_to_gguf.py ./smolvlm-screen-merged (from llama.cpp)')\n